# Classifier comparison and feature medians

This notebook does two things:

1. It compares the two classifiers of notebooks 10a (method A – class weights) and 10b
   (method B – equal classes): EER, scores per codec and weights of the features.
2. It shows the typical (median) value of every feature for bona fide and spoofed recordings in
   every data set – per data set and subset, per codec and per source. These are the
   differences (shortcuts) the classifiers can use.

**How to use it:** run notebooks 09, 10a and 10b first, then the cells from top to bottom.

**Output files**

In `results/classifier/10c_classifier_comparison_and_feature_medians_output/` (in git):

- `comparison_eer.csv`, `comparison_score_medians.csv`, `comparison_coefficients.csv` (step 2),
- `feature_medians.csv` (step 4), `feature_medians_per_codec.csv` (step 5),
  `feature_medians_per_source.csv` (step 6).

**Steps**

1. Setup
2. Comparison of notebooks 10a and 10b
3. Feature tables
4. Feature medians per data set, subset and class
5. Feature medians per codec
6. Feature medians per source
7. Saving the results

### Step 1 – Setup

**What it does:** Sets the paths, checks that the results of notebooks 10a and 10b exist and
lists the feature tables of notebook 09 (all but SONICS).

**Why:** All the next cells use these paths, and a missing file is visible at once.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from xai_audio_deepfake_shortcuts_audit.classifier.config import (
    BREAKDOWNS,
    CODEC_REFERENCES,
    FEATURE_COLUMNS,
    SUFFIX,
)
from xai_audio_deepfake_shortcuts_audit.classifier.select_features import select_features
from xai_audio_deepfake_shortcuts_audit.features.read_feature_table import read_feature_table
from xai_audio_deepfake_shortcuts_audit.utils.find_repo_root import find_repo_root

REPO_ROOT = find_repo_root(Path.cwd())
FEATURES_DIR = REPO_ROOT / "outputs" / "09_feature_extraction_output" / "features"
NOTEBOOK_NAME = "10c_classifier_comparison_and_feature_medians"
CLASSIFIER_RESULTS_DIR = REPO_ROOT / "results" / "classifier"
RESULTS_DIR = CLASSIFIER_RESULTS_DIR / f"{NOTEBOOK_NAME}_output"
RESULTS_10A_DIR = CLASSIFIER_RESULTS_DIR / "10a_feature_classifier_all_data_class_weights_output"
RESULTS_10B_DIR = CLASSIFIER_RESULTS_DIR / "10b_feature_classifier_equal_classes_output"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILES = sorted(
    path for path in FEATURES_DIR.glob(f"features_*_{SUFFIX}.csv") if "05_sonics" not in path.name
)

print("Results of notebook 10a exist:", (RESULTS_10A_DIR / "eer_table.csv").is_file())
print("Results of notebook 10b exist:", (RESULTS_10B_DIR / "eer_table.csv").is_file())
display(pd.DataFrame({"feature_file": [path.name for path in FEATURE_FILES]}))

### Step 2 – Comparison of notebooks 10a and 10b

**What it does:** Reads the results of notebooks 10a (A) and 10b (B) and shows them next to
each other: the EER with the difference B − A in percentage points (pp; 16.60 % → 16.71 % is
+0.11 pp), the scores per codec and the weights of the features.

**Why:** Both models differ only in how the classes were balanced, so this shows what that
choice changes. A negative EER difference means fewer errors with equal classes.

In [ ]:
KEYS = ["dataset", "subset", "group", "value"]
MEDIAN_KEYS = ["dataset", "subset", "source", "codec"]
MEDIAN_COLUMNS = [
    "median_score_bonafide_with_codec",
    "shift_bonafide",
    "median_score_spoof_with_codec",
]
COEFFICIENT_COLUMNS = ["feature", "coefficient", "towards"]

eer_10a = pd.read_csv(RESULTS_10A_DIR / "eer_table.csv", dtype={key: str for key in KEYS})
eer_10b = pd.read_csv(RESULTS_10B_DIR / "eer_table.csv", dtype={key: str for key in KEYS})
comparison_eer = eer_10a[[*KEYS, "eer"]].merge(
    eer_10b[[*KEYS, "eer"]], on=KEYS, suffixes=("_10a", "_10b")
)
comparison_eer["difference_pp"] = (comparison_eer["eer_10b"] - comparison_eer["eer_10a"]) * 100

medians_10a = pd.read_csv(
    RESULTS_10A_DIR / "score_medians.csv", dtype={key: str for key in MEDIAN_KEYS}
)
medians_10b = pd.read_csv(
    RESULTS_10B_DIR / "score_medians.csv", dtype={key: str for key in MEDIAN_KEYS}
)
comparison_medians = medians_10a[[*MEDIAN_KEYS, *MEDIAN_COLUMNS]].merge(
    medians_10b[[*MEDIAN_KEYS, *MEDIAN_COLUMNS]], on=MEDIAN_KEYS, suffixes=("_10a", "_10b")
)

coefficients_10a = pd.read_csv(RESULTS_10A_DIR / "coefficients.csv")
coefficients_10b = pd.read_csv(RESULTS_10B_DIR / "coefficients.csv")
comparison_coefficients = coefficients_10a[COEFFICIENT_COLUMNS].merge(
    coefficients_10b[COEFFICIENT_COLUMNS], on="feature", suffixes=("_10a", "_10b")
)
comparison_coefficients["difference"] = (
    comparison_coefficients["coefficient_10b"] - comparison_coefficients["coefficient_10a"]
)

for group in ["all", "attack_id", "codec", "source"]:
    print("EER of every test set:" if group == "all" else f"EER per {group}:")
    display(
        comparison_eer[comparison_eer["group"] == group]
        .style.format("{:+.2f} pp", subset=["difference_pp"])
        .format("{:.2%}", subset=["eer_10a", "eer_10b"])
        .hide(axis="index")
    )
print("Scores per codec:")
display(comparison_medians.style.format(precision=3).hide(axis="index"))
print("Weights of the features:")
display(comparison_coefficients.style.format(precision=3).hide(axis="index"))

### Step 3 – Feature tables

**What it does:** Reads the feature tables of notebook 09 (all but SONICS), keeps the usable
recordings (`select_features`: no error, no missing feature) and counts them per data set,
subset and class.

**Why:** The next steps compute the medians from these recordings; the counts show how many
recordings each median is based on.

In [ ]:
usable_tables = []
for path in FEATURE_FILES:
    table = read_feature_table(path)
    features, _ = select_features(table)
    usable_tables.append(table.loc[features.index])
usable = pd.concat(usable_tables, ignore_index=True)

display(usable.groupby(["dataset", "subset", "label"]).size().unstack("label", fill_value=0))
print(f"Usable recordings: {len(usable):,}")

### Step 4 – Feature medians per data set, subset and class

**What it does:** For every data set, subset and class (bona fide, spoof) shows the number of
recordings and the median of each of the five features.

**Why:** A large difference between bona fide and spoof in one feature is a shortcut: a model
can use it to tell the classes apart without listening to the speech.

In [ ]:
AGGREGATIONS = {
    "recordings": ("label", "size"),
    **{column: (column, "median") for column in FEATURE_COLUMNS},
}
FORMATS = {
    "duration_s": "{:.2f}",
    f"leading_silence_s_{SUFFIX}": "{:.3f}",
    f"trailing_silence_s_{SUFFIX}": "{:.3f}",
    "average_loudness_dbfs": "{:.2f}",
    "bandwidth_hz": "{:.0f}",
}

feature_medians = usable.groupby(["dataset", "subset", "label"]).agg(**AGGREGATIONS).reset_index()
display(feature_medians.style.format(FORMATS).hide(axis="index"))

### Step 5 – Feature medians per codec

**What it does:** The same as step 4, but for every codec of the test sets that are split per
codec in notebooks 10a and 10b (2021 LA eval, 2021 DF eval), separately for every source – as
in the scores per codec of 10a and 10b. All 2021 LA recordings come from ASVspoof 2019 LA eval,
so their source is shown as `asvspoof`. The recordings without a codec come first.

**Why:** It shows how each codec changes the features of bona fide and spoofed recordings,
e.g. how much bandwidth is left after a telephone codec. Comparing inside one source keeps the
other differences between the sources out.

In [ ]:
CODEC_SETS = [key for key, columns in BREAKDOWNS.items() if "codec" in columns]
in_codec_sets = pd.MultiIndex.from_frame(usable[["dataset", "subset"]]).isin(CODEC_SETS)

# 2021 LA has no source column: all its recordings come from ASVspoof 2019 LA eval
codec_recordings = usable[in_codec_sets].assign(
    source=lambda table: table["source"].replace("", "asvspoof")
)
feature_medians_per_codec = (
    codec_recordings.groupby(["dataset", "subset", "source", "codec", "label"])
    .agg(**AGGREGATIONS)
    .reset_index()
)

is_reference = [
    codec == CODEC_REFERENCES.get((dataset, subset))
    for dataset, subset, codec in zip(
        feature_medians_per_codec["dataset"],
        feature_medians_per_codec["subset"],
        feature_medians_per_codec["codec"],
        strict=True,
    )
]
feature_medians_per_codec = (
    feature_medians_per_codec.assign(is_reference=is_reference)
    .sort_values(
        ["dataset", "subset", "source", "is_reference", "codec", "label"],
        ascending=[True, True, True, False, True, True],
    )
    .drop(columns="is_reference")
)

for (dataset, subset, source), table in feature_medians_per_codec.groupby(
    ["dataset", "subset", "source"], sort=False
):
    print(f"{dataset} {subset}, source {source} – feature medians per codec:")
    display(table.style.format(FORMATS).hide(axis="index"))

### Step 6 – Feature medians per source

**What it does:** The same as step 4, but for every source corpus of the test sets that are
split per source in notebooks 10a and 10b (2021 DF eval: asvspoof, vcc2018, vcc2020).

**Why:** It shows whether the shortcuts of ASVspoof 2019 (e.g. longer leading silence of bona
fide recordings) also exist in the recordings from other corpora.

In [ ]:
SOURCE_SETS = [key for key, columns in BREAKDOWNS.items() if "source" in columns]
in_source_sets = pd.MultiIndex.from_frame(usable[["dataset", "subset"]]).isin(SOURCE_SETS)

feature_medians_per_source = (
    usable[in_source_sets]
    .groupby(["dataset", "subset", "source", "label"])
    .agg(**AGGREGATIONS)
    .reset_index()
)
display(feature_medians_per_source.style.format(FORMATS).hide(axis="index"))

### Step 7 – Saving the results

**What it does:** Saves the comparison of step 2 and the feature medians of steps 4–6 to
`results/`.

**Why:** The tables are small and go to git, so they can be read and cited without running
the notebook again.

In [ ]:
results = {
    "comparison_eer.csv": comparison_eer,
    "comparison_score_medians.csv": comparison_medians,
    "comparison_coefficients.csv": comparison_coefficients,
    "feature_medians.csv": feature_medians,
    "feature_medians_per_codec.csv": feature_medians_per_codec,
    "feature_medians_per_source.csv": feature_medians_per_source,
}
for name, table in results.items():
    table.to_csv(RESULTS_DIR / name, index=False)
    print("Saved:", (RESULTS_DIR / name).relative_to(REPO_ROOT))